# Model scope and assumptions

- **Valuation date:** December 31, 2025. The fixed $100 million book holds $60 million of the CRSP value-weighted S&P 500 universe index and $40 million of modeled Treasury positions. Each historical move is applied to this same book, then the book is reset.
- **Modeled bonds:** Four $100-face fixed-rate bonds mature in 2, 5, 10, and 30 years and pay semiannual coupons. Each coupon rate equals its December 31, 2025 maturity-matched Treasury benchmark yield. That yield is used as the bond's YTM proxy for pricing and shocked repricing. The benchmark files do not identify actual bonds or CUSIPs. Repricing is instantaneous, with no coupon accrual or passage of time.
- **Historical risk:** Treasury yield changes are in basis points and are paired with CRSP daily total returns only when both observations cover the same start and end dates. Standalone and combined daily 99% VaR use the nearest-rank loss; 97.5% ES averages the worst ceiling of 2.5% of sample days. Historical event days are separate from Fed workbook shocks.
- **Fed stress:** The workbook's United States equity spot shock is a proxy for the CRSP index. USD Governments 2Y, 5Y, 10Y, and 30Y shocks are applied to modeled bond yields, followed by full cash-flow repricing. The 2026 scenario is applied retrospectively because its workbook was generated after the valuation date. Equity and bond P&Ls add, so there is no separate cross-asset P&L term; offset and amplification describe their joint outcomes.
- **Illustrative limits:** $2.5 million daily 99% VaR, $2.75 million daily 97.5% ES, and $25 million loss for each Fed scenario. These are project assumptions, not institutional limits. Capacity scales the complete 60/40 book uniformly.


In [1]:
import pandas as pd


In [2]:
equity = pd.read_csv("lbgq5t2hrjhtpw9t.csv", parse_dates=["DlyCalDt"])
rates = {
    tenor: pd.read_csv(f"{tenor}.csv", parse_dates=["observation_date"])
    for tenor in ["DGS2", "DGS5", "DGS10", "DGS30"]
}

for name, df in {"equity": equity, **rates}.items():
    date_col = "DlyCalDt" if name == "equity" else "observation_date"
    values = df.drop(columns=[date_col, "INDNO"], errors="ignore")
    numeric = values.apply(pd.to_numeric, errors="coerce")
    print(f"\n{name}: {len(df)} rows; columns={df.columns.tolist()}")
    print("Date range:", df[date_col].min(), "to", df[date_col].max())
    print("Missing dates:", df[date_col].isna().sum())
    print("Duplicate dates:", df[date_col].duplicated().sum())
    print("Dates sorted:", df[date_col].is_monotonic_increasing)
    print("Missing/nonnumeric:", numeric.isna().sum().to_dict())
    print("Value ranges:\n", numeric.agg(["min", "max"]))

yield_panel = pd.concat(
    [df.set_index("observation_date")[name] for name, df in rates.items()],
    axis=1,
).apply(pd.to_numeric, errors="coerce")
eq_dates = pd.DatetimeIndex(
    equity.loc[pd.to_numeric(equity["DlyTotRet"], errors="coerce").notna(), "DlyCalDt"]
).dropna().unique()
rate_dates = yield_panel.dropna().index
matched = eq_dates.intersection(rate_dates).sort_values()



equity: 5031 rows; columns=['INDNO', 'DlyCalDt', 'DlyTotRet', 'DlyPrcRet']
Date range: 2006-01-03 00:00:00 to 2025-12-31 00:00:00
Missing dates: 0
Duplicate dates: 0
Dates sorted: True
Missing/nonnumeric: {'DlyTotRet': 0, 'DlyPrcRet': 0}
Value ranges:
      DlyTotRet  DlyPrcRet
min  -0.118970  -0.119008
max   0.115133   0.115133

DGS2: 5217 rows; columns=['observation_date', 'DGS2']
Date range: 2006-01-03 00:00:00 to 2025-12-31 00:00:00
Missing dates: 0
Duplicate dates: 0
Dates sorted: True
Missing/nonnumeric: {'DGS2': 214}
Value ranges:
      DGS2
min  0.09
max  5.29

DGS5: 5217 rows; columns=['observation_date', 'DGS5']
Date range: 2006-01-03 00:00:00 to 2025-12-31 00:00:00
Missing dates: 0
Duplicate dates: 0
Dates sorted: True
Missing/nonnumeric: {'DGS5': 214}
Value ranges:
      DGS5
min  0.19
max  5.23

DGS10: 5217 rows; columns=['observation_date', 'DGS10']
Date range: 2006-01-03 00:00:00 to 2025-12-31 00:00:00
Missing dates: 0
Duplicate dates: 0
Dates sorted: True
Missing/nonnu

In [3]:
print("\nIndex IDs:", equity["INDNO"].unique())
print("Complete matched dates:", len(matched), matched.min(), matched.max())
print("Equity dates without complete yields:", eq_dates.difference(rate_dates).tolist())
print("Complete yield dates without equity:", rate_dates.difference(eq_dates).tolist())
print("Valuation-date yields:\n", yield_panel.loc["2025-12-31"])


Index IDs: [1000500]
Complete matched dates: 4994 2006-01-03 00:00:00 2025-12-31 00:00:00
Equity dates without complete yields: [Timestamp('2006-10-09 00:00:00'), Timestamp('2007-10-08 00:00:00'), Timestamp('2007-11-12 00:00:00'), Timestamp('2008-10-13 00:00:00'), Timestamp('2008-11-11 00:00:00'), Timestamp('2009-10-12 00:00:00'), Timestamp('2009-11-11 00:00:00'), Timestamp('2010-10-11 00:00:00'), Timestamp('2010-11-11 00:00:00'), Timestamp('2011-10-10 00:00:00'), Timestamp('2011-11-11 00:00:00'), Timestamp('2012-10-08 00:00:00'), Timestamp('2012-11-12 00:00:00'), Timestamp('2013-10-14 00:00:00'), Timestamp('2013-11-11 00:00:00'), Timestamp('2014-10-13 00:00:00'), Timestamp('2014-11-11 00:00:00'), Timestamp('2015-10-12 00:00:00'), Timestamp('2015-11-11 00:00:00'), Timestamp('2016-10-10 00:00:00'), Timestamp('2016-11-11 00:00:00'), Timestamp('2017-10-09 00:00:00'), Timestamp('2018-10-08 00:00:00'), Timestamp('2018-11-12 00:00:00'), Timestamp('2019-10-14 00:00:00'), Timestamp('2019-11-1

In [4]:
workbooks = {
    "2020": "ccar-2020-severely-adverse-market-shocks.xlsx",
    "2022": "2022-stress-test-severely-adverse-market-shocks.xlsx",
    "2026": "2026_Final_Severely_Adverse_Market_Shock_simplified-shocks.xlsx",
}

for year, path in workbooks.items():
    book = pd.ExcelFile(path)
    print(f"\n{year} sheets: {book.sheet_names}")
    for sheet in book.sheet_names:
        raw = pd.read_excel(book, sheet_name=sheet, header=None)
        print(f"\nSheet: {sheet!r}; dimensions: {raw.shape}")
        for row_id, row in raw.dropna(how="all").head(8).iterrows():
            cells = {
                int(col): str(value)
                for col, value in row.items()
                if pd.notna(value)
            }
            print(f"Row {row_id}: {cells}")


2020 sheets: ['Cover Sheet', 'Equity by Geography', 'Dividends', 'FX Spot', 'FX Vega', 'Rates DV01', 'Rates Vega-Normal & Relative', 'Rates Vega-Normal & Absolute', 'Other Rates', 'Energy', 'Metals', 'Ags & Softs', 'Commodity Indices', 'Securitized Products', 'Agencies', 'Munis', 'ARS', 'Corporate Credit-Advanced', 'Corporate Credit-EM', 'Sovereign Credit', 'Credit Correlation', 'Private Equity', 'Other Fair Value Assets']

Sheet: 'Cover Sheet'; dimensions: (54, 3)
Row 1: {1: 'Trading, PE and Other Fair Value Assets: Market Shocks'}
Row 3: {1: 'Scenario Name:', 2: 'CCAR 2020: Severely Adverse'}
Row 5: {1: 'As-Of Date:', 2: '2019-10-18 00:00:00'}
Row 7: {1: 'Date of Generation:', 2: '2020-02-10 00:00:00'}
Row 53: {1: 'CCAR 2020: Severely Adverse Scenario'}

Sheet: 'Equity by Geography'; dimensions: (79, 18)
Row 0: {0: 'CCAR 2020: Severely Adverse Scenario'}
Row 1: {0: 'Equity by Geography'}
Row 4: {5: 'Vol Points'}
Row 5: {1: 'Country', 3: 'Spot Shock (%)', 5: '1M', 6: '3M', 7: '6M', 8

In [5]:
from openpyxl import load_workbook

for year, path in workbooks.items():
    wb = load_workbook(path, data_only=True)
    print(f"\n--- {year} ---")

    for sheet in ["Equity by Geography", "Rates DV01"]:
        ws = wb[sheet]
        print(f"\n{sheet}")
        selected = set()

        for row in ws.iter_rows():
            labels = [str(c.value).strip().upper() for c in row
                      if c.value is not None]
            if any(s in {"US", "USA", "UNITED STATES", "U.S."}
                   or s.startswith("USD") for s in labels):
                selected.update(range(max(1, row[0].row - 1),
                                      min(ws.max_row, row[0].row + 4) + 1))

        if not selected:
            print("No US/USD label found.")
        for row_number in sorted(selected):
            print([
                (c.coordinate, c.value, c.number_format)
                for c in ws[row_number] if c.value is not None
            ])
    wb.close()


--- 2020 ---

Equity by Geography
[('B27', 'United Kingdom', 'General'), ('D27', -0.113, '0.0%'), ('F27', 10.7, '#,##0.0_);[Red]\\(#,##0.0\\)'), ('G27', 9.6, '#,##0.0_);[Red]\\(#,##0.0\\)'), ('H27', 8.3, '#,##0.0_);[Red]\\(#,##0.0\\)'), ('I27', 7.4, '#,##0.0_);[Red]\\(#,##0.0\\)'), ('J27', 6.7, '#,##0.0_);[Red]\\(#,##0.0\\)'), ('K27', 5.2, '#,##0.0_);[Red]\\(#,##0.0\\)'), ('L27', 4.6, '#,##0.0_);[Red]\\(#,##0.0\\)'), ('M27', 4.1, '#,##0.0_);[Red]\\(#,##0.0\\)'), ('N27', 3.8, '#,##0.0_);[Red]\\(#,##0.0\\)'), ('O27', 3.7, '#,##0.0_);[Red]\\(#,##0.0\\)'), ('P27', 3.6, '#,##0.0_);[Red]\\(#,##0.0\\)'), ('Q27', 3.5, '#,##0.0_);[Red]\\(#,##0.0\\)'), ('R27', 3.4, '#,##0.0_);[Red]\\(#,##0.0\\)')]
[('B28', 'United States', 'General'), ('D28', -0.26, '0.0%'), ('F28', 32.6, '#,##0.0_);[Red]\\(#,##0.0\\)'), ('G28', 30.7, '#,##0.0_);[Red]\\(#,##0.0\\)'), ('H28', 28.5, '#,##0.0_);[Red]\\(#,##0.0\\)'), ('I28', 26.8, '#,##0.0_);[Red]\\(#,##0.0\\)'), ('J28', 25.4, '#,##0.0_);[Red]\\(#,##0.0\\)'), ('K28

In [6]:
face = 100.0
terms = {2: 3.47, 5: 3.73, 10: 4.18, 30: 4.84}

def bond_price(face, coupon_pct, yield_pct, years):
    periods = int(2 * years)
    coupon = face * (coupon_pct / 100) / 2
    period_rate = (yield_pct / 100) / 2

    return sum(
        (coupon + (face if period == periods else 0))
        / (1 + period_rate) ** period
        for period in range(1, periods + 1)
    )

for years, yield_pct in terms.items():
    price = bond_price(face, yield_pct, yield_pct, years)
    print(f"{years:>2}Y: price per $100 face = ${price:.8f}")
    assert abs(price - 100) < 1e-8

 2Y: price per $100 face = $100.00000000
 5Y: price per $100 face = $100.00000000
10Y: price per $100 face = $100.00000000
30Y: price per $100 face = $100.00000000


In [7]:
one_bp_in_percent = 0.01
dv01_per_100 = {}

for years, yield_pct in terms.items():
    price_down = bond_price(face, yield_pct, yield_pct - one_bp_in_percent, years)
    price_up = bond_price(face, yield_pct, yield_pct + one_bp_in_percent, years)
    dv01 = (price_down - price_up) / 2
    dv01_per_100[years] = dv01

    print(
        f"{years:>2}Y: price at yield -1 bp = ${price_down:.8f}, "
        f"price at yield +1 bp = ${price_up:.8f}, "
        f"DV01 = ${dv01:.8f} per $100 face"
    )
    assert dv01 > 0

 2Y: price at yield -1 bp = $100.01916405, price at yield +1 bp = $99.98084062, DV01 = $0.01916171 per $100 face
 5Y: price at yield -1 bp = $100.04524370, price at yield +1 bp = $99.95478005, DV01 = $0.04523182 per $100 face
10Y: price at yield -1 bp = $100.08109048, price at yield +1 bp = $99.91898743, DV01 = $0.08105152 per $100 face
30Y: price at yield -1 bp = $100.15758109, price at yield +1 bp = $99.84278103, DV01 = $0.15740003 per $100 face


In [8]:
treasury_budget = 40_000_000.0
target_dv01 = treasury_budget / (
    100 * sum(1 / dv01 for dv01 in dv01_per_100.values())
)

positions = {}
for years, dv01 in dv01_per_100.items():
    units = target_dv01 / dv01
    market_value = units * bond_price(
        face, terms[years], terms[years], years
    )
    positions[years] = {
        "units": units,
        "face_value": units * face,
        "market_value": market_value,
        "dollar_dv01": units * dv01,
    }
    print(
        f"{years:>2}Y: face=${units * face:,.2f}, "
        f"MV=${market_value:,.2f}, "
        f"DV01=${units * dv01:,.2f}/bp"
    )

print("Total Treasury MV:", sum(p["market_value"] for p in positions.values()))
assert abs(sum(p["market_value"] for p in positions.values()) - treasury_budget) < 0.01
assert max(p["dollar_dv01"] for p in positions.values()) - min(
    p["dollar_dv01"] for p in positions.values()
) < 0.01

 2Y: face=$22,449,380.42, MV=$22,449,380.42, DV01=$4,301.69/bp
 5Y: face=$9,510,308.42, MV=$9,510,308.42, DV01=$4,301.69/bp
10Y: face=$5,307,347.35, MV=$5,307,347.35, DV01=$4,301.69/bp
30Y: face=$2,732,963.80, MV=$2,732,963.80, DV01=$4,301.69/bp
Total Treasury MV: 40000000.0


In [9]:
complete_yields = yield_panel.dropna().sort_index()
yield_changes_bp = complete_yields.diff().dropna() * 100
rate_start = pd.Series(complete_yields.index, index=complete_yields.index).shift()

equity_returns = (
    equity.set_index("DlyCalDt")["DlyTotRet"]
    .pipe(pd.to_numeric, errors="coerce")
    .sort_index()
    .rename("equity_return")
)
valid_equity = equity_returns.dropna()
equity_start = pd.Series(valid_equity.index, index=valid_equity.index).shift()

ending_date_matches = yield_changes_bp.join(valid_equity, how="inner").dropna()
same_interval = (
    rate_start.reindex(ending_date_matches.index)
    == equity_start.reindex(ending_date_matches.index)
)
joint_moves = ending_date_matches.loc[same_interval].copy()

print("End-date matches:", len(ending_date_matches))
print("Dropped for differing start dates:", (~same_interval).sum())
print("Joint one-interval moves:", len(joint_moves))
print("Date range:", joint_moves.index.min(), "to", joint_moves.index.max())
print("Missing values:", joint_moves.isna().sum().to_dict())
print(joint_moves.head(3))
print(joint_moves.tail(3))
assert same_interval.reindex(joint_moves.index).all()

End-date matches: 4993
Dropped for differing start dates: 46
Joint one-interval moves: 4947
Date range: 2006-01-04 00:00:00 to 2025-12-31 00:00:00
Missing values: {'DGS2': 0, 'DGS5': 0, 'DGS10': 0, 'DGS30': 0, 'equity_return': 0}
            DGS2  DGS5  DGS10  DGS30  equity_return
2006-01-04  -3.0  -2.0   -1.0    3.0       0.003931
2006-01-05   1.0   1.0    0.0   -4.0       0.000028
2006-01-06   4.0   3.0    2.0    2.0       0.009559
            DGS2  DGS5  DGS10  DGS30  equity_return
2025-12-29  -1.0  -1.0   -2.0   -1.0      -0.003518
2025-12-30   0.0   1.0    2.0    1.0      -0.001398
2025-12-31   2.0   5.0    4.0    3.0      -0.007304


In [10]:
equity_value = 60_000_000.0

for date, move in joint_moves.iloc[[0, -1]].iterrows():
    equity_pnl = equity_value * move["equity_return"]
    treasury_pnl = 0.0

    for years in terms:
        shock_bp = move[f"DGS{years}"]
        shocked_yield_pct = terms[years] + shock_bp / 100
        shocked_price = bond_price(
            face, terms[years], shocked_yield_pct, years
        )
        starting_price = bond_price(
            face, terms[years], terms[years], years
        )
        treasury_pnl += positions[years]["units"] * (
            shocked_price - starting_price
        )

    print(
        f"{date.date()}: equity=${equity_pnl:,.2f}, "
        f"Treasury=${treasury_pnl:,.2f}, "
        f"total=${equity_pnl + treasury_pnl:,.2f}"
    )

2006-01-04: equity=$235,860.00, Treasury=$12,960.79, total=$248,820.79
2025-12-31: equity=$-438,240.00, Treasury=$-60,115.83, total=$-498,355.83


In [11]:
def pnl_for_move(move):
    equity_pnl = equity_value * move["equity_return"]
    treasury_pnl = sum(
        positions[years]["units"] * (
            bond_price(
                face, terms[years],
                terms[years] + move[f"DGS{years}"] / 100,
                years
            ) - 100.0
        )
        for years in terms
    )
    return pd.Series({
        "equity_pnl": equity_pnl,
        "treasury_pnl": treasury_pnl,
        "total_pnl": equity_pnl + treasury_pnl,
    })

historical_pnl = joint_moves.apply(pnl_for_move, axis=1)

print("Scenarios:", len(historical_pnl))
print("Missing values:", historical_pnl.isna().sum().to_dict())
print(historical_pnl.iloc[[0, -1]].round(2))
print(
    "Largest daily loss:",
    historical_pnl["total_pnl"].min(),
    "on",
    historical_pnl["total_pnl"].idxmin().date()
)

Scenarios: 4947
Missing values: {'equity_pnl': 0, 'treasury_pnl': 0, 'total_pnl': 0}
            equity_pnl  treasury_pnl  total_pnl
2006-01-04    235860.0      12960.79  248820.79
2025-12-31   -438240.0     -60115.83 -498355.83
Largest daily loss: -6803021.018937003 on 2020-03-16


In [12]:
import math

def historical_var_es(pnl):
    losses = (-pnl).sort_values(ascending=False)
    var_count = math.ceil(0.01 * len(losses))
    es_count = math.ceil(0.025 * len(losses))
    return {
        "99% VaR": losses.iloc[var_count - 1],
        "97.5% ES": losses.iloc[:es_count].mean(),
        "VaR rank": var_count,
        "ES days": es_count,
    }

for name, column in [
    ("Equity", "equity_pnl"),
    ("Treasury", "treasury_pnl"),
    ("Combined", "total_pnl"),
]:
    result = historical_var_es(historical_pnl[column])
    print(
        f"{name:>8}: 99% VaR=${result['99% VaR']:,.2f}, "
        f"97.5% ES=${result['97.5% ES']:,.2f}, "
        f"VaR rank={result['VaR rank']}, ES days={result['ES days']}"
    )

  Equity: 99% VaR=$2,111,880.00, 97.5% ES=$2,319,714.19, VaR rank=50, ES days=124
Treasury: 99% VaR=$226,883.99, 97.5% ES=$232,777.12, VaR rank=50, ES days=124
Combined: 99% VaR=$2,092,052.60, 97.5% ES=$2,225,975.37, VaR rank=50, ES days=124


In [13]:
shock_2020 = {
    "equity": -0.26,
    2: -77.7, 5: -66.6, 10: -60.2, 30: -55.5
}

equity_scenario_pnl = equity_value * shock_2020["equity"]
bond_scenario_pnl = {}

for years in terms:
    shocked_yield_pct = terms[years] + shock_2020[years] / 100
    shocked_price = bond_price(
        face, terms[years], shocked_yield_pct, years
    )
    bond_scenario_pnl[years] = positions[years]["units"] * (
        shocked_price - 100.0
    )
    print(
        f"{years:>2}Y: shocked yield={shocked_yield_pct:.3f}%, "
        f"price=${shocked_price:.6f}, "
        f"P&L=${bond_scenario_pnl[years]:,.2f}"
    )

treasury_scenario_pnl = sum(bond_scenario_pnl.values())
print(f"Equity P&L:   ${equity_scenario_pnl:,.2f}")
print(f"Treasury P&L: ${treasury_scenario_pnl:,.2f}")
print(f"Total P&L:    ${equity_scenario_pnl + treasury_scenario_pnl:,.2f}")
scenario_results = {
    "2020": {
        "equity_pnl": equity_scenario_pnl,
        "treasury_pnl": treasury_scenario_pnl,
        "total_pnl": equity_scenario_pnl + treasury_scenario_pnl,
    }
}

 2Y: shocked yield=2.693%, price=$101.503065, P&L=$337,428.72
 5Y: shocked yield=3.064%, price=$103.065788, P&L=$291,565.87
10Y: shocked yield=3.578%, price=$105.023488, P&L=$266,613.95
30Y: shocked yield=4.285%, price=$109.321770, P&L=$254,760.59
Equity P&L:   $-15,600,000.00
Treasury P&L: $1,150,369.13
Total P&L:    $-14,449,630.87


In [14]:
remaining_shocks = {
    "2022": {"equity": -0.383, 2: 29.9, 5: 42.3, 10: 49.4, 30: 54.7},
    "2026": {"equity": -0.150, 2: 88.8, 5: 90.2, 10: 91.0, 30: 91.6},
}

for year, shock in remaining_shocks.items():
    equity_pnl = equity_value * shock["equity"]
    treasury_pnl = 0.0
    print(f"\n{year} scenario")

    for years in terms:
        shocked_yield_pct = terms[years] + shock[years] / 100
        shocked_price = bond_price(
            face, terms[years], shocked_yield_pct, years
        )
        bond_pnl = positions[years]["units"] * (shocked_price - 100.0)
        treasury_pnl += bond_pnl
        print(f"{years:>2}Y: P&L=${bond_pnl:,.2f}")

    print(f"Equity P&L:   ${equity_pnl:,.2f}")
    print(f"Treasury P&L: ${treasury_pnl:,.2f}")
    print(f"Total P&L:    ${equity_pnl + treasury_pnl:,.2f}")
    scenario_results[year] = {
        "equity_pnl": equity_pnl,
        "treasury_pnl": treasury_pnl,
        "total_pnl": equity_pnl + treasury_pnl,
    }


2022 scenario
 2Y: P&L=$-128,153.32
 5Y: P&L=$-179,957.39
10Y: P&L=$-207,542.75
30Y: P&L=$-221,182.74
Equity P&L:   $-22,980,000.00
Treasury P&L: $-736,836.20
Total P&L:    $-23,716,836.20

2026 scenario
 2Y: P&L=$-377,893.41
 5Y: P&L=$-378,983.51
10Y: P&L=$-374,858.52
30Y: P&L=$-355,658.51
Equity P&L:   $-9,000,000.00
Treasury P&L: $-1,487,393.94
Total P&L:    $-10,487,393.94


In [15]:
scenario_interaction = pd.DataFrame.from_dict(scenario_results, orient="index")
scenario_interaction.index.name = "Fed scenario"
scenario_interaction["effect_on_equity_loss_pct"] = (
    scenario_interaction["treasury_pnl"].abs()
    / scenario_interaction["equity_pnl"].abs() * 100
)
scenario_interaction["effect"] = scenario_interaction["treasury_pnl"].apply(
    lambda x: "offset" if x > 0 else "amplification"
)
print(scenario_interaction.to_string(float_format=lambda x: f"{x:,.2f}"))

                 equity_pnl  treasury_pnl      total_pnl  effect_on_equity_loss_pct         effect
Fed scenario                                                                                      
2020         -15,600,000.00  1,150,369.13 -14,449,630.87                       7.37         offset
2022         -22,980,000.00   -736,836.20 -23,716,836.20                       3.21  amplification
2026          -9,000,000.00 -1,487,393.94 -10,487,393.94                      16.53  amplification


In [16]:
equity_risk = historical_var_es(historical_pnl["equity_pnl"])
treasury_risk = historical_var_es(historical_pnl["treasury_pnl"])
combined_risk = historical_var_es(historical_pnl["total_pnl"])

for measure in ["99% VaR", "97.5% ES"]:
    standalone_sum = equity_risk[measure] + treasury_risk[measure]
    benefit = standalone_sum - combined_risk[measure]
    print(
        f"{measure}: standalone sum=${standalone_sum:,.2f}, "
        f"combined=${combined_risk[measure]:,.2f}, "
        f"benefit=${benefit:,.2f} "
        f"({benefit / standalone_sum:.2%})"
    )

99% VaR: standalone sum=$2,338,763.99, combined=$2,092,052.60, benefit=$246,711.40 (10.55%)
97.5% ES: standalone sum=$2,552,491.32, combined=$2,225,975.37, benefit=$326,515.95 (12.79%)


In [17]:
limits_and_usage = {
    "99% VaR": (2_500_000, combined_risk["99% VaR"]),
    "97.5% ES": (2_750_000, combined_risk["97.5% ES"]),
    "Fed 2020": (25_000_000, -scenario_results["2020"]["total_pnl"]),
    "Fed 2022": (25_000_000, -scenario_results["2022"]["total_pnl"]),
    "Fed 2026": (25_000_000, -scenario_results["2026"]["total_pnl"]),
}

for name, (limit, usage) in limits_and_usage.items():
    print(
        f"{name:>9}: limit=${limit:,.2f}, "
        f"usage=${usage:,.2f}, "
        f"headroom=${limit - usage:,.2f}"
    )

  99% VaR: limit=$2,500,000.00, usage=$2,092,052.60, headroom=$407,947.40
 97.5% ES: limit=$2,750,000.00, usage=$2,225,975.37, headroom=$524,024.63
 Fed 2020: limit=$25,000,000.00, usage=$14,449,630.87, headroom=$10,550,369.13
 Fed 2022: limit=$25,000,000.00, usage=$23,716,836.20, headroom=$1,283,163.80
 Fed 2026: limit=$25,000,000.00, usage=$10,487,393.94, headroom=$14,512,606.06


In [18]:
scale_by_limit = {
    name: limit / usage
    for name, (limit, usage) in limits_and_usage.items()
}

for name, scale in scale_by_limit.items():
    print(f"{name:>9}: maximum scale={scale:.4f}x")

binding = min(scale_by_limit, key=scale_by_limit.get)
max_scale = scale_by_limit[binding]

print("Binding limit:", binding)
print(f"Additional portfolio size: ${(max_scale - 1) * 100_000_000:,.2f}")
print(f"Equity at limit: ${60_000_000 * max_scale:,.2f}")
print(f"Treasuries at limit: ${40_000_000 * max_scale:,.2f}")

  99% VaR: maximum scale=1.1950x
 97.5% ES: maximum scale=1.2354x
 Fed 2020: maximum scale=1.7301x
 Fed 2022: maximum scale=1.0541x
 Fed 2026: maximum scale=2.3838x
Binding limit: Fed 2022
Additional portfolio size: $5,410,349.81
Equity at limit: $63,246,209.89
Treasuries at limit: $42,164,139.92


In [19]:
yield_step = 0.01       # one basis point, in yield-percent units
decimal_step = 0.0001  # the same move as a decimal yield

for years, coupon_pct in terms.items():
    p0 = bond_price(face, coupon_pct, coupon_pct, years)
    p_down = bond_price(face, coupon_pct, coupon_pct - yield_step, years)
    p_up = bond_price(face, coupon_pct, coupon_pct + yield_step, years)

    convexity = (p_down + p_up - 2 * p0) / (
        p0 * decimal_step**2
    )
    position_curvature = positions[years]["units"] * (
        p_down + p_up - 2 * p0
    )

    print(
        f"{years:>2}Y: convexity={convexity:.4f}, "
        f"position price curvature=${position_curvature:,.4f} per bp²"
    )
    assert convexity > 0

 2Y: convexity=4.6682, position price curvature=$1.0480 per bp²
 5Y: convexity=23.7455, position price curvature=$2.2583 per bp²
10Y: convexity=77.9171, position price curvature=$4.1353 per bp²
30Y: convexity=362.1194, position price curvature=$9.8966 per bp²


In [20]:
march_2020 = pd.Timestamp("2020-03-16")
pnl_2022 = historical_pnl.loc["2022"]
rates_2022 = pnl_2022["treasury_pnl"].idxmin()

for label, date in [
    ("March 2020 selloff", march_2020),
    ("2022 worst Treasury day", rates_2022),
]:
    print(f"\n{label}: {date.date()}")
    print("Market move:", joint_moves.loc[date].to_dict())
    print("P&L:", historical_pnl.loc[date].round(2).to_dict())


March 2020 selloff: 2020-03-16
Market move: {'DGS2': -13.0, 'DGS5': -20.999999999999996, 'DGS10': -20.999999999999996, 'DGS30': -21.999999999999996, 'equity_return': -0.11897}
P&L: {'equity_pnl': -7138200.0, 'treasury_pnl': 335178.98, 'total_pnl': -6803021.02}

2022 worst Treasury day: 2022-06-13
Market move: {'DGS2': 33.999999999999986, 'DGS5': 31.000000000000007, 'DGS10': 28.000000000000025, 'DGS30': 21.999999999999975, 'equity_return': -0.038909}
P&L: {'equity_pnl': -2334540.0, 'treasury_pnl': -489056.71, 'total_pnl': -2823596.71}


In [21]:
for label, date in [
    ("March 2020 selloff", march_2020),
    ("2022 worst Treasury day", rates_2022),
]:
    move = joint_moves.loc[date]
    equity_part = equity_value * move["equity_return"]
    bond_parts = {}

    for years in terms:
        shocked_yield = terms[years] + move[f"DGS{years}"] / 100
        bond_parts[years] = positions[years]["units"] * (
            bond_price(face, terms[years], shocked_yield, years) - 100
        )

    attributed_total = equity_part + sum(bond_parts.values())
    reported_total = historical_pnl.loc[date, "total_pnl"]

    print(f"\n{label}: {date.date()}")
    print("Equity:", round(equity_part, 2))
    print("Treasury by tenor:", {k: round(v, 2) for k, v in bond_parts.items()})
    print("Attributed total:", round(attributed_total, 2))
    print("Unattributed residual:", round(reported_total - attributed_total, 8))


March 2020 selloff: 2020-03-16
Equity: -7138200.0
Treasury by tenor: {2: np.float64(56010.58), 5: np.float64(90835.38), 10: np.float64(91253.91), 30: np.float64(97079.1)}
Attributed total: -6803021.02
Unattributed residual: 0.0

2022 worst Treasury day: 2022-06-13
Equity: -2334540.0
Treasury by tenor: {2: np.float64(-145653.59), 5: np.float64(-132273.64), 10: np.float64(-118841.77), 30: np.float64(-92287.71)}
Attributed total: -2823596.71
Unattributed residual: 0.0


In [22]:
without_diversification = dict(limits_and_usage)
without_diversification["99% VaR"] = (
    2_500_000,
    equity_risk["99% VaR"] + treasury_risk["99% VaR"],
)
without_diversification["97.5% ES"] = (
    2_750_000,
    equity_risk["97.5% ES"] + treasury_risk["97.5% ES"],
)

for name in ["99% VaR", "97.5% ES"]:
    limit, combined_usage = limits_and_usage[name]
    _, standalone_sum = without_diversification[name]
    extra_capacity = 100_000_000 * (
        limit / combined_usage - limit / standalone_sum
    )
    print(f"{name}: added capacity from diversification=${extra_capacity:,.2f}")

scale_without = min(
    limit / usage for limit, usage in without_diversification.values()
)
scale_with = min(
    limit / usage for limit, usage in limits_and_usage.values()
)
print(f"All limits, without benefit: {scale_without:.4f}x")
print(f"All limits, with benefit:    {scale_with:.4f}x")
print(f"Added capacity under all limits: ${(scale_with - scale_without) * 100_000_000:,.2f}")

99% VaR: added capacity from diversification=$12,605,794.51
97.5% ES: added capacity from diversification=$15,803,470.36
All limits, without benefit: 1.0541x
All limits, with benefit:    1.0541x
Added capacity under all limits: $0.00


In [23]:
worst_equity_days = historical_pnl.nsmallest(math.ceil(0.025 * len(historical_pnl)), "equity_pnl")
treasury_on_worst = worst_equity_days["treasury_pnl"]

print(
    "Equity–Treasury daily P&L correlation:",
    historical_pnl["equity_pnl"].corr(historical_pnl["treasury_pnl"])
)
print(
    f"Average Treasury P&L on {len(worst_equity_days)} worst equity days:",
    treasury_on_worst.mean()
)
print(
    "Treasury gains / losses on those days:",
    (treasury_on_worst > 0).sum(),
    "/",
    (treasury_on_worst < 0).sum()
)

Equity–Treasury daily P&L correlation: -0.30535431388745926
Average Treasury P&L on 124 worst equity days: 104215.42313365654
Treasury gains / losses on those days: 100 / 24
